In [3]:
import pandas as pd
import requests
from pathlib import Path

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

URL = "https://data.cityofnewyork.us/resource/h9gi-nx95.csv"
CHUNK = 50_000
MAX   = 500_000

print(f"Downloading {MAX:,} rows...")

chunks, offset = [], 0

while offset < MAX:
    params = {"$limit": CHUNK, "$offset": offset, "$order": "crash_date DESC"}
    r = requests.get(URL, params=params, timeout=60)
    chunk = pd.read_csv(pd.io.common.StringIO(r.text))
    if len(chunk) == 0:
        break
    chunks.append(chunk)
    offset += CHUNK
    print(f"  {offset:,} rows done", flush=True)
df = pd.concat(chunks, ignore_index=True)
df.to_csv(RAW / "crashes_raw.csv", index=False)
print(f"Saved. Total rows: {len(df):,}")

  50,000 rows done
  100,000 rows done
  150,000 rows done
  200,000 rows done
  250,000 rows done
  300,000 rows done
  350,000 rows done
  400,000 rows done
  450,000 rows done
  500,000 rows done
Saved. Total rows: 500,000


In [4]:
df = pd.read_csv("../data/raw/crashes_raw.csv", low_memory=False)

print(f"Shape: {df.shape}")
print("\nMissing values (columns with nulls only):")
missing = df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))

Shape: (500000, 29)

Missing values (columns with nulls only):
vehicle_type_code_5              496489
contributing_factor_vehicle_5    496339
vehicle_type_code_4              488502
contributing_factor_vehicle_4    487846
vehicle_type_code_3              457341
contributing_factor_vehicle_3    454067
cross_street_name                358628
off_street_name                  248123
vehicle_type_code2               169560
zip_code                         149459
borough                          149388
on_street_name                   141377
contributing_factor_vehicle_2    118221
location                          32633
longitude                         32633
latitude                          32633
vehicle_type_code1                 7672
contributing_factor_vehicle_1      3236
dtype: int64


In [5]:
total = len(df)
no_lat    = df['latitude'].isnull().sum()
zero_coords = ((df['latitude'] == 0) | (df['longitude'] == 0)).sum()
usable    = df['latitude'].notnull() & (df['latitude'] != 0) & (df['longitude'] != 0)

print(f"Total rows:       {total:,}")
print(f"Missing lat/lon:  {no_lat:,}  ({no_lat/total*100:.1f}%)")
print(f"Zero coords:      {zero_coords:,}  ({zero_coords/total*100:.1f}%)")
print(f"Usable for join:  {usable.sum():,}  ({usable.sum()/total*100:.1f}%)")

print("\nBorough distribution:")
print(df['borough'].value_counts(dropna=False))

df['crash_date'] = pd.to_datetime(df['crash_date'])
print(f"\nDate range: {df['crash_date'].min().date()} → {df['crash_date'].max().date()}")

print("\nInjury/fatality summary:")
print(df[['number_of_persons_injured', 'number_of_persons_killed']].describe().round(2))

Total rows:       500,000
Missing lat/lon:  32,633  (6.5%)
Zero coords:      5,942  (1.2%)
Usable for join:  461,425  (92.3%)

Borough distribution:
borough
NaN              149388
BROOKLYN         122166
QUEENS            94003
MANHATTAN         62489
BRONX             57943
STATEN ISLAND     14011
Name: count, dtype: int64

Date range: 2021-02-01 → 2026-03-31

Injury/fatality summary:
       number_of_persons_injured  number_of_persons_killed
count                  500000.00                 500000.00
mean                        0.54                      0.00
std                         0.84                      0.06
min                         0.00                      0.00
25%                         0.00                      0.00
50%                         0.00                      0.00
75%                         1.00                      0.00
max                        40.00                      5.00
